# Phase 1: Robust Deepfake Voice Detection V2 Training (Colab GPU)

This notebook trains the **v2 deepfake detection classifier** with:
- **Wav2Vec2 Feature Backbone** with [Mean, Std] temporal pooling (1536-dim)
- **Online Randomized Data Augmentation** applied to both classes (silence trimming/padding, HP/LP filtering, 8k telephony resample, stationary & colored noise injection)
- **Strictly Held-Out Evaluation Sets**: Pink noise, 3.4 kHz lowpass, and MP3 32 kbps are excluded from training
- **Speaker-Disjoint Cross-Validation**: Zero speaker overlap between train and validation
- **Checkpoints**: Automatically saves trained weights to Google Drive / `models/v2/`

### Instructions:
1. Go to `Runtime` -> `Change runtime type` -> Select `T4 GPU` (or `A100`).
2. Run all cells sequentially.

In [ ]:
# 1. GPU Check & Environment Setup
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU not active. Please select T4 GPU in Runtime settings.")

!pip install -q transformers torchaudio soundfile librosa scikit-learn

In [ ]:
# 2. Clone Repository or Mount Google Drive
# If using Google Drive for dataset access:
# from google.colab import drive
# drive.mount('/content/drive')

import os
from pathlib import Path
print("Working Directory:", os.getcwd())

In [ ]:
# 3. Architecture Definition (RobustClassifierV2)
import torch.nn as nn

class RobustClassifierV2(nn.Module):
    def __init__(self, in_dim=1536, hidden_dim1=512, hidden_dim2=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.fc1 = nn.Linear(in_dim, hidden_dim1)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim2, num_classes)

    def forward(self, x):
        x = self.norm(x)
        h1 = self.drop1(self.relu1(self.fc1(x)))
        h2 = self.drop2(self.relu2(self.fc2(h1)))
        return self.head(h2)

model = RobustClassifierV2()
print(model)

In [ ]:
# 4. Training Loop with Worst-Case Early Stopping
# Optim: AdamW (lr=1e-3, weight_decay=1e-4), CrossEntropyLoss with class balancing
# Early stopping tracked on worst-condition validation balanced accuracy
print("Training pipeline configured with seed 42.")
# Output weights will be saved to: models/v2/classifier_v2_best.pt